# HR Assistant Agent: Ground Truth 평가

이 Notebook에서는 Amazon Bedrock AgentCore Evaluations를 사용해 Ground Truth를 기반으로 에이전트 애플리케이션을 평가하는 방법을 보여 줍니다.

| 인터페이스 | 사용 시점 |
|---|---|
| **EvaluationClient** | CloudWatch에 에이전트 세션이 이미 있으며, 특정 세션을 참조 입력과 비교해 평가하려는 경우 |
| **OnDemandEvaluationDatasetRunner** | 테스트 데이터 세트가 있으며, 각 시나리오에 대해 에이전트를 호출하고 결과를 평가하려는 경우 |
| **BatchEvaluationRunner** | 단일 API 호출로 여러 세션을 한 번에 평가하고 evaluator별 집계 점수를 얻으려는 경우 |

**다른 평가 유형과 비교**

| 항목 | 온디맨드 | 온라인 | 배치 |
|---|---|---|---|
| **트리거** | 호출자가 시작, 동기식 | 연속 실행, 이벤트 기반 | 호출자가 시작, 비동기식 |
| **세션 소스** | 호출자가 span을 인라인으로 제공 | 로그 그룹 모니터링 | 서비스가 CloudWatch Logs에서 검색 |
| **범위** | 단일 세션 | 샘플링 규칙과 일치하는 모든 세션 | 여러 세션(시간 범위, 세션 ID 또는 전체 로그 그룹) |
| **Ground Truth** | `evaluationReferenceInputs`를 통해 제공 | 지원되지 않음 | 인라인 Ground Truth가 포함된 `sessionMetadata`를 통해 제공 |
| **결과** | 동기식 응답 | CloudWatch 지표 및 대시보드 | evaluator별 평균이 포함된 집계 요약 및 CloudWatch의 세션별 세부 정보 |
| **사용 사례** | 개발 중 표본 검사, CI/CD | 프로덕션 모니터링 | 기준선 측정, 변경 전후 비교, 회귀 테스트 |

actor LLM을 사용하는 시뮬레이션 멀티턴 평가 예제는 `03-advanced/02-simulating-agent-interactions/Strands-AgentCore-ShoppingConcierge.ipynb`를 참조하세요.

### HR Assistant Agent

Acme Corp 직원을 지원하는 Strands agent인 **HR Assistant**를 배포합니다. 이 에이전트는 다음 작업을 수행합니다.
- PTO 잔여 일수 확인 및 휴가 요청
- HR 정책 조회(PTO, 원격 근무, 육아 휴직)
- 복리후생 정보 확인(건강, 치과, 시력, 401k)
- 급여 명세서 조회

### 학습 내용
- `EvaluationClient`를 사용해 Amazon CloudWatch에 기록된 기존 에이전트 세션을 Ground Truth 참조와 함께 평가하는 방법
- `OnDemandEvaluationDatasetRunner`를 사용해 자동 데이터 세트 평가를 실행하는 방법
- 기본 제공 evaluator(Correctness, GoalSuccessRate, Trajectory)의 평가 결과를 해석하는 방법
- `StartBatchEvaluation`으로 **배치 평가**를 실행해 여러 세션의 점수를 한 번에 산출하는 방법

### 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|---|---|
| Agent 프레임워크 | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation SDK | `bedrock-agentcore` |
| AWS 서비스 | AgentCore Runtime, AgentCore Evaluations, CloudWatch Logs |

### 사전 요구 사항
- Python 3.10+
- AgentCore, CloudWatch, CodeBuild, ECR, IAM 권한이 있는 AWS 자격 증명

## 1단계: 종속성 설치

In [ ]:
!pip install -r requirements.txt -q --upgrade

## 2단계: 구성

라이브러리를 가져오고 AWS 세션을 구성합니다.

In [ ]:
import boto3
import json
import time
import uuid
from datetime import timedelta
from boto3.session import Session
from IPython.display import display, Markdown

region = "aws_region"  # 여기에 AWS 리전을 입력합니다
boto_session = Session(region_name=region)
REGION = boto_session.region_name

print(f"Region  : {REGION}")

## 3단계: HR Assistant Agent 배포

배포 로직은 이 디렉터리의 `deploy_hr_assistant_agent.py`에 있습니다. 이 스크립트는 `agentcore` CLI를 사용해 다음 작업을 수행합니다.

1. `agentcore configure`로 에이전트 **구성**(entrypoint, 이름, 리전, requirements)
2. CodeBuild 이미지 빌드, ECR push, OTel 계측 및 Runtime 생성을 자동으로 처리하는 `agentcore deploy`로 **배포**
3. **READY 상태가 될 때까지 폴링**

이 스크립트는 Notebook 네임스페이스에 `AGENT_ID`, `AGENT_ARN`, `CW_LOG_GROUP`, `agentcore_client`를 설정합니다.


HR Assistant agent 소스는 이 디렉터리의 `hr_assistant_agent.py`에 있습니다.
이 [Strands](https://strandsagents.com/) agent는 결정론적 mock data를 사용하는 다섯 가지 도구를 갖추고 있어
평가 결과를 완전히 재현할 수 있습니다.

| 도구 | 설명 |
|---|---|
| `get_pto_balance` | Remaining PTO days for an employee |
| `submit_pto_request` | Request time off |
| `lookup_hr_policy` | Company policy documents (PTO, remote work, parental leave, code of conduct) |
| `get_benefits_summary` | Health, dental, vision, 401k, life insurance details |
| `get_pay_stub` | Pay stub for a given period |

에이전트는 `us.amazon.nova-lite-v1:0`을 모델로 사용하며, 멀티턴을 지원하기 위해
세션 ID별로 대화 기록을 캐시합니다.

In [ ]:
# HR Assistant agent를 배포합니다.
# 전체 배포 흐름은 deploy_hr_assistant_agent.py를 참조하세요.
%run -i deploy_hr_assistant_agent.py

# _REGION은 아래의 사용자 지정 evaluator 셀에서 사용됩니다
_REGION = REGION

print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")

## 3단계: 에이전트를 호출해 세션 생성

평가하려면 CloudWatch span이 포함된 에이전트 세션이 필요합니다. 여러 시나리오에서 에이전트를
호출하고 `EvaluationClient`에서 사용할 세션 ID를 기록합니다.

각 세션은 하나의 평가 시나리오에 해당합니다.

In [ ]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """HR Assistant에 단일 프롬프트를 보내고 응답 텍스트를 반환합니다."""
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


def run_session(turns: list[str], session_prefix: str) -> str:
    """멀티턴 세션을 호출하고 해당 세션 ID를 반환합니다."""
    session_id = f"{session_prefix}-{uuid.uuid4()}"
    print(f"Session: {session_id}")
    for turn_input in turns:
        print(f"  > {turn_input[:70]}")
        response = invoke_agent(turn_input, session_id)
        print(f"  < {response[:100]}")
    return session_id

In [ ]:
# --- 싱글턴 세션 ---

print("=== Single-Turn Sessions ===")

session_pto_balance = run_session(["What is the current PTO balance for employee EMP-001?"], "pto-balance-check")

session_submit_pto = run_session(
    ["Please submit a PTO request for employee EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation."],
    "submit-pto-request",
)

session_pay_stub = run_session(
    ["Can you pull up the January 2026 pay stub for employee EMP-001?"],
    "pay-stub-lookup",
)

print("\nSingle-turn sessions created.")

In [ ]:
# --- 멀티턴 세션: PTO 계획 ---

print("=== Multi-Turn Session: PTO Planning ===")

session_pto_planning = run_session(
    [
        "How many PTO days do I have left? My employee ID is EMP-001.",
        "Great. I'd like to take December 23 to December 25 off. Please submit a request.",
        "Remind me, what is the policy on rolling over unused PTO?",
    ],
    "pto-planning-session",
)

print("\nMulti-turn session created.")

In [ ]:
# --- 멀티턴 세션: 신규 직원 온보딩 ---

print("=== Multi-Turn Session: New Employee Onboarding ===")

session_onboarding = run_session(
    [
        "I just joined the company. What is the remote work policy?",
        "How much PTO do I get as a new employee?",
        "What life insurance benefit does the company provide?",
        "Can you check the current PTO balance for employee EMP-042?",
    ],
    "new-employee-onboarding",
)

print("\nAll sessions created. Waiting 60s for CloudWatch log ingestion...")
time.sleep(60)
print("Ready to evaluate.")

## 5단계: EvaluationClient - 기존 세션 평가

`EvaluationClient`는 CloudWatch에 기록된 **기존 에이전트 세션**을 Ground Truth와 임시로 비교해 테스트할 때 적합합니다.
지정한 `session_id`에 해당하는 에이전트 span을 조회하고 evaluator를 실행합니다. 이러한 평가에는 `expected_response`, `assertions`, `expected_trajectory`를 전달할 수 있습니다. 기본 제공 evaluator와 사용자 지정 evaluator를 모두 사용할 수 있습니다.

### Ground Truth 참조 입력

`ReferenceInputs`를 사용하면 선택적으로 Ground Truth를 제공할 수 있습니다.

| 필드 | 사용하는 evaluator | 설명 |
|---|---|---|
| `expected_response` | `Builtin.Correctness` | 이상적인 응답 텍스트 |
| `expected_trajectory` | `Builtin.TrajectoryExactOrderMatch`, `Builtin.TrajectoryInOrderMatch`, `Builtin.TrajectoryAnyOrderMatch` | 순서가 지정된 도구 이름 목록 |
| `assertions` | `Builtin.GoalSuccessRate` | 세션이 충족해야 하는 자유 형식 assertion |

Ground Truth가 필요하지 않은 evaluator(`Helpfulness`, `ResponseRelevance`)도 같은 호출에 포함할 수 있습니다.
각 evaluator는 필요한 필드만 읽습니다.

## 사용자 지정 LLM-as-a-Judge evaluator 생성

기본 제공 evaluator 외에도 **사용자 지정 LLM-as-a-Judge evaluator**를 사용해 자체 평가 기준을
정의할 수 있습니다. 이러한 evaluator는 평가 시 자동으로 대체되는 **Ground Truth placeholder**를
참조할 수 있는 자연어 지침을 받습니다.

### Ground Truth placeholder

| 수준 | 사용 가능한 placeholder |
|---|---|
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` |
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` |

예를 들어 응답 유사성을 비교하는 TRACE 수준 evaluator는 지침에 `{assistant_turn}`과
`{expected_response}`를 포함합니다. evaluator가 실행되면 서비스가 해당 placeholder를
실제 에이전트 출력 및 `ReferenceInputs`의 `expectedResponse`로 대체합니다.

### 생성할 항목

| Evaluator | 수준 | Placeholder | 설명 |
|---|---|---|---|
| `HRResponseSimilarity` | TRACE | `{assistant_turn}`, `{expected_response}` | 에이전트 응답이 예상 답변과 얼마나 유사한지 평가 |
| `HRAssertionChecker` | SESSION | `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | 에이전트가 올바른 도구를 호출하고 모든 세션 assertion을 충족했는지 평가 |

In [ ]:
_SUFFIX = uuid.uuid4().hex[:8]
_cp = boto3.client("bedrock-agentcore-control", region_name=_REGION)

# ---------------------------------------------------------------------------
# TRACE 수준: HRResponseSimilarity
# 에이전트 응답을 expected_response 참조 입력과 비교합니다.
# {assistant_turn} → 실제 에이전트 출력
# {expected_response} → ReferenceInputs의 expectedResponse 필드
# ---------------------------------------------------------------------------
print("Creating HRResponseSimilarity (TRACE) ...")
_resp_sim = _cp.create_evaluator(
    evaluatorName=f"HRResponseSimilarity_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Compare the agent's response with the expected response.\n"
                "Agent response: {assistant_turn}\n"
                "Expected response: {expected_response}\n\n"
                "Rate how closely the agent's response matches the expected response. "
                "Focus on whether the key facts, numbers, and conclusions agree."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "not_similar",
                        "definition": "Response is factually different or missing key information from the expected response.",
                    },
                    {
                        "value": 0.5,
                        "label": "partially_similar",
                        "definition": "Response captures some expected content but omits or misrepresents parts.",
                    },
                    {
                        "value": 1.0,
                        "label": "highly_similar",
                        "definition": "Response is semantically equivalent to the expected response — all key facts match.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_RESPONSE_SIMILARITY_ID = _resp_sim["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_RESPONSE_SIMILARITY_ID}")

# ---------------------------------------------------------------------------
# SESSION 수준: HRAssertionChecker
# 도구 trajectory 준수 여부와 assertion 충족 여부를 평가합니다.
# {actual_tool_trajectory}   → 에이전트가 실제로 호출한 도구
# {expected_tool_trajectory} → ReferenceInputs의 expectedTrajectory
# {assertions}               → ReferenceInputs의 assertions 목록
# ---------------------------------------------------------------------------
print("\nCreating HRAssertionChecker (SESSION) ...")
_assert_chk = _cp.create_evaluator(
    evaluatorName=f"HRAssertionChecker_{_SUFFIX}",
    level="SESSION",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Evaluate whether the agent fulfilled the session requirements.\n\n"
                "Expected tool trajectory: {expected_tool_trajectory}\n"
                "Actual tool trajectory: {actual_tool_trajectory}\n"
                "Assertions to verify: {assertions}\n\n"
                "Score the agent on how well it followed the expected tool trajectory "
                "and satisfied every listed assertion."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "failed",
                        "definition": "Agent did not follow the trajectory and failed most assertions.",
                    },
                    {
                        "value": 0.5,
                        "label": "partial",
                        "definition": "Agent partially followed the trajectory or satisfied only some assertions.",
                    },
                    {
                        "value": 1.0,
                        "label": "passed",
                        "definition": "Agent followed the expected trajectory and satisfied all assertions.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_ASSERTION_CHECKER_ID = _assert_chk["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_ASSERTION_CHECKER_ID}")

print("\nCustom evaluators ready:")
print(f"  HRResponseSimilarity (TRACE)   : {CUSTOM_RESPONSE_SIMILARITY_ID}")
print(f"  HRAssertionChecker   (SESSION) : {CUSTOM_ASSERTION_CHECKER_ID}")

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

eval_client = EvaluationClient(region_name=REGION)

print(f"EvaluationClient initialised (region={REGION})")
print(f"  {CUSTOM_RESPONSE_SIMILARITY_ID} → TRACE  (custom: HRResponseSimilarity)")
print(f"  {CUSTOM_ASSERTION_CHECKER_ID} → SESSION (custom: HRAssertionChecker)")

In [ ]:
# 출력 도우미 함수
def display_eval_results(label: str, results: list) -> None:
    """EvaluationClient 결과를 Markdown 표로 보기 좋게 출력합니다."""
    rows = ["| Evaluator | Value | Label | Explanation |", "|---|---|---|---|"]
    for r in results:
        evaluator = r.get("evaluatorId", "")[:40]
        value = str(r.get("value", r.get("score", "N/A")))
        lbl = str(r.get("label", r.get("rating", "")))
        explanation = (r.get("explanation", r.get("reason", "")) or "")[:120].replace("\n", " ")
        error_code = r.get("errorCode")
        if error_code:
            lbl = f"ERR:{error_code}"
            explanation = (r.get("errorMessage", "") or "")[:120]
        rows.append(f"| `{evaluator}` | {value} | {lbl} | {explanation} |")

    if len(rows) == 2:  # 헤더 행만 있고 데이터는 없음
        rows.append("| No results — session may be too recent or spans not yet visible | | | |")

    md = f"### {label}\n\n" + "\n".join(rows)
    display(Markdown(md))

### 5a. 싱글턴: PTO 잔여 일수 - Correctness + Helpfulness + 사용자 지정 ResponseSimilarity

`Builtin.Correctness`와 사용자 지정 `HRResponseSimilarity` evaluator(`{assistant_turn}` 및
`{expected_response}` placeholder 사용)를 사용해 PTO 잔여 일수 응답을 알려진 예상 답변과
비교합니다. 둘 다 사실 정확성을 측정하지만 서로 다른 점수 rubric을 사용합니다.

In [ ]:
pto_balance_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",  # TRACE: 제공된 expected response와 비교
        "Builtin.Helpfulness",  # TRACE: Ground Truth 불필요
        "Builtin.ResponseRelevance",  # TRACE: Ground Truth 불필요
        CUSTOM_RESPONSE_SIMILARITY_ID,  # TRACE: 사용자 지정, {assistant_turn} + {expected_response} 사용
    ],
    session_id=session_pto_balance,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
    ),
)

display_eval_results(
    "PTO Balance — Correctness + Quality + Custom ResponseSimilarity",
    pto_balance_results,
)

### 5b. 싱글턴: PTO 제출 - Assertions + Trajectory + 사용자 지정 AssertionChecker

이 셀은 기본 제공 trajectory evaluator와 사용자 지정 `HRAssertionChecker`
(`{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` placeholder 사용)를
모두 실행하며, 응답에는 사용자 지정 `HRResponseSimilarity`도 적용합니다. 이를 통해 기본 제공
점수와 사용자 지정 점수를 나란히 비교할 수 있습니다.

In [ ]:
submit_pto_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",  # SESSION: 기본 제공 assertion evaluator
        "Builtin.TrajectoryExactOrderMatch",  # SESSION: 기본 제공 trajectory evaluator
        "Builtin.TrajectoryAnyOrderMatch",  # SESSION: 기본 제공 trajectory evaluator
        "Builtin.Correctness",  # TRACE: 기본 제공 응답 정확도 evaluator
        CUSTOM_RESPONSE_SIMILARITY_ID,  # TRACE(사용자 지정): {assistant_turn} + {expected_response}
    ],
    session_id=session_submit_pto,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["submit_pto_request"],
        assertions=[
            "Agent called submit_pto_request for employee EMP-001",
            "Agent confirmed the PTO request was approved",
            "Agent provided a request ID (e.g. PTO-2026-001)",
        ],
        expected_response="PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
    ),
)

display_eval_results("PTO Submission — Built-in + Custom ResponseSimilarity", submit_pto_results)

### 5c. 싱글턴: 급여 명세서 - 사실 정확성

사실 데이터 검색 시나리오에는 `Builtin.Correctness`와 `Builtin.GoalSuccessRate`의 조합이
적합합니다. expected_response는 Ground Truth 수치를 제공합니다.

In [ ]:
pay_stub_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",
        "Builtin.GoalSuccessRate",
    ],
    session_id=session_pay_stub,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
        assertions=[
            "Agent called get_pay_stub for EMP-001 period 2026-01",
            "Agent reported the correct gross pay of $8,333.33",
            "Agent reported the correct net pay of $5,362.50",
        ],
    ),
)

display_eval_results("Pay Stub Lookup — Correctness + GoalSuccessRate", pay_stub_results)

### 5d. 멀티턴: PTO 계획 세션(3턴) + 사용자 지정 AssertionChecker

멀티턴 세션에서 `EvaluationClient`는 세션의 모든 span을 가져와 전체 대화를 평가합니다.
trajectory와 assertions는 모든 턴에 적용됩니다.

이 시나리오에서는 `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}`
placeholder를 사용하는 SESSION 수준의 사용자 지정 `HRAssertionChecker` evaluator도 실행합니다.
각 턴마다 서로 다른 도구를 호출하는 3턴 세션은 evaluator가 예상 순서와 비교할 수 있는
풍부한 trajectory를 제공합니다.

In [ ]:
pto_planning_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.TrajectoryExactOrderMatch",
        "Builtin.TrajectoryInOrderMatch",
        "Builtin.TrajectoryAnyOrderMatch",
        "Builtin.Helpfulness",
        CUSTOM_ASSERTION_CHECKER_ID,  # SESSION(사용자 지정): {actual_tool_trajectory} + {expected_tool_trajectory} + {assertions}
    ],
    session_id=session_pto_planning,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=[
            "get_pto_balance",
            "submit_pto_request",
            "lookup_hr_policy",
        ],
        assertions=[
            "Agent correctly reported 10 remaining PTO days for EMP-001 in turn 1",
            "Agent submitted a PTO request for December 23-25, 2026 in turn 2",
            "Agent correctly stated the 5-day PTO rollover limit in turn 3",
        ],
    ),
)

display_eval_results(
    "PTO Planning — Multi-Turn (3 turns) + Custom AssertionChecker",
    pto_planning_results,
)

## 6단계: OnDemandEvaluationDatasetRunner - 자동 데이터 세트 평가

`OnDemandEvaluationDatasetRunner`는 **테스트 데이터 세트**를 사용해 다음 작업을 수행할 때 적합합니다.
1. 각 시나리오에서 에이전트 자동 호출
2. CloudWatch span 수집
3. 각 시나리오 결과에 evaluator 실행

선별된 데이터 세트에 대한 회귀 테스트, CI/CD 파이프라인 및 배치 평가에 적합합니다.

### 데이터 세트 구조

데이터 세트는 하나 이상의 **턴**을 포함하는 **시나리오**로 구성됩니다. 선택적 Ground Truth 필드는 다음과 같습니다.
- `Turn.expected_response` - 턴별 예상 답변
- `PreDefinedScenario.expected_trajectory` - 순서가 지정된 도구 이름 목록
- `PreDefinedScenario.assertions` - 세션 수준 assertions

### OnDemandEvaluationDatasetRunner 작동 방식

```
각 시나리오에서:
  1. 새 세션 ID 생성
  2. 각 턴에 대해 agent_invoker 함수 호출
  3. CloudWatch span이 나타날 때까지 대기(evaluation_delay_seconds)
  4. span과 Ground Truth를 평가 서비스에 제출
  5. 결과를 수집해 반환
```

In [ ]:
from bedrock_agentcore.evaluation import (
    AgentInvokerInput,
    AgentInvokerOutput,
    CloudWatchAgentSpanCollector,
    Dataset,
    EvaluationRunConfig,
    OnDemandEvaluationDatasetRunner,
    EvaluatorConfig,
    Turn,
    PredefinedScenario,
)

In [ ]:
def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """
    OnDemandEvaluationDatasetRunner가 턴마다 한 번씩 호출합니다. HR Assistant를
    호출하고 응답 텍스트를 반환합니다.

    AgentInvokerInput 필드:
      - payload:    데이터 세트에서 가져온 턴 입력(str 또는 dict).
      - session_id: 프레임워크가 관리하며 시나리오의 모든 턴에서 유지되는 세션 ID.
                    대화의 연속성을 위해 에이전트에 전달합니다.
    """
    payload = invoker_input.payload
    body = {"prompt": payload} if isinstance(payload, str) else payload

    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=invoker_input.session_id,
        payload=json.dumps(body).encode("utf-8"),
    )

    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return AgentInvokerOutput(agent_output="".join(parts) if parts else raw)

### 6a. 평가 데이터 세트 정의

시나리오를 인라인으로 정의합니다. 싱글턴 및 멀티턴 시나리오를 함께 사용해
에이전트의 여러 측면을 테스트합니다.

In [ ]:
dataset = Dataset(
    scenarios=[
        # --- 싱글턴: PTO 잔여 일수 ---
        PredefinedScenario(
            scenario_id="pto-balance-check",
            turns=[
                Turn(
                    input="What is the current PTO balance for employee EMP-001?",
                    expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
                )
            ],
            expected_trajectory=["get_pto_balance"],
            assertions=[
                "Agent called get_pto_balance with employee_id=EMP-001",
                "Agent reported 10 remaining PTO days",
            ],
        ),
        # --- 싱글턴: HR 정책 조회 ---
        PredefinedScenario(
            scenario_id="pto-policy-lookup",
            turns=[
                Turn(
                    input="What is the company PTO policy?",
                    expected_response="Full-time employees accrue 15 days of PTO per year. Requests must be submitted at least 2 business days in advance. Up to 5 unused days roll over each year.",
                )
            ],
            expected_trajectory=["lookup_hr_policy"],
            assertions=[
                "Agent called lookup_hr_policy with topic=pto",
                "Agent mentioned the 15-day annual accrual for full-time employees",
                "Agent mentioned the 2 business day advance notice requirement",
            ],
        ),
        # --- 싱글턴: 401k 복리후생 ---
        PredefinedScenario(
            scenario_id="401k-info",
            turns=[
                Turn(
                    input="How does the 401k match work?",
                    expected_response="The company matches 100% of contributions up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%. The match vests over 3 years.",
                )
            ],
            expected_trajectory=["get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary with benefit_type=401k",
                "Agent correctly described the 4% full match and 50% match on next 2%",
                "Agent mentioned the 3-year vesting schedule",
            ],
        ),
        # --- 싱글턴: 잔여 일수 확인 후 PTO 제출 ---
        PredefinedScenario(
            scenario_id="check-and-submit-pto",
            turns=[
                Turn(
                    input="Check the PTO balance for EMP-002, and if they have at least 2 days, submit a request for 2026-05-26 to 2026-05-27.",
                    expected_response="EMP-002 has 3 remaining PTO days. PTO request submitted and approved for 2026-05-26 to 2026-05-27.",
                )
            ],
            expected_trajectory=["get_pto_balance", "submit_pto_request"],
            assertions=[
                "Agent first called get_pto_balance for EMP-002",
                "Agent confirmed 3 remaining days is sufficient",
                "Agent then called submit_pto_request for the correct dates",
            ],
        ),
        # --- 멀티턴: 복리후생 탐색 ---
        PredefinedScenario(
            scenario_id="benefits-exploration",
            turns=[
                Turn(
                    input="Can you walk me through the health insurance options?",
                    expected_response="The company covers 90% of premiums for employee-only coverage. Three plans are available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA.",
                ),
                Turn(
                    input="What about dental?",
                    expected_response="The dental plan covers 100% of preventive care, 80% of basic restorative care, and 50% of major work, with a $2,000 annual maximum.",
                ),
                Turn(
                    input="And how much does the company contribute to the 401k?",
                    expected_response="The company matches 100% up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%.",
                ),
            ],
            expected_trajectory=[
                "get_benefits_summary",
                "get_benefits_summary",
                "get_benefits_summary",
            ],
            assertions=[
                "Agent called get_benefits_summary three times across the conversation",
                "Agent correctly described health, dental, and 401k benefits in their respective turns",
                "Agent maintained conversational context across all three turns",
            ],
        ),
    ]
)

print(f"Dataset contains {len(dataset.scenarios)} scenarios.")

### 6b. OnDemandEvaluationDatasetRunner 구성 및 실행

In [ ]:
# Span 수집기: 에이전트가 내보낸 OTel span을 CloudWatch에서 폴링합니다
span_collector = CloudWatchAgentSpanCollector(
    log_group_name=CW_LOG_GROUP,
    region=REGION,
    max_wait_seconds=180,
    poll_interval_seconds=15,
)

# Evaluator 수준 캐시 - 기본 제공 + 사용자 지정 evaluator
EVALUATOR_LEVELS = {
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.TrajectoryExactOrderMatch": "SESSION",
    "Builtin.TrajectoryInOrderMatch": "SESSION",
    "Builtin.TrajectoryAnyOrderMatch": "SESSION",
    "Builtin.Correctness": "TRACE",
}
# 사용자 지정 evaluator(HR Response Similarity와 HRAssertionChecker용으로 생성한 evaluator)
EVALUATOR_LEVELS[CUSTOM_RESPONSE_SIMILARITY_ID] = "TRACE"
EVALUATOR_LEVELS[CUSTOM_ASSERTION_CHECKER_ID] = "SESSION"

# Evaluator 구성 - 기본 제공 evaluator와 사용자 지정 evaluator 조합
config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.Correctness",  # TRACE - expected_response 참조
            "Builtin.GoalSuccessRate",  # SESSION - assertions 참조
            "Builtin.TrajectoryExactOrderMatch",  # SESSION - expected_trajectory 참조
            "Builtin.TrajectoryInOrderMatch",  # SESSION - expected_trajectory 참조
            "Builtin.TrajectoryAnyOrderMatch",  # SESSION - expected_trajectory 참조
            CUSTOM_RESPONSE_SIMILARITY_ID,  # TRACE(사용자 지정) - {assistant_turn} + {expected_response}
            CUSTOM_ASSERTION_CHECKER_ID,  # SESSION(사용자 지정) - {actual_tool_trajectory} + {assertions}
        ]
    ),
    evaluation_delay_seconds=180,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=REGION)
runner._evaluator_level_cache.update(EVALUATOR_LEVELS)

print("OnDemandEvaluationDatasetRunner configured. Starting evaluation...")
print(f"  Scenarios : {len(dataset.scenarios)}")
print(f"  Evaluators: {len(config.evaluator_config.evaluator_ids)} (7 built-in + 2 custom)")
print(f"  Delay     : {config.evaluation_delay_seconds}s (waiting for CloudWatch ingestion)")

In [ ]:
# 평가를 실행합니다.
# OnDemandEvaluationDatasetRunner는 다음을 수행합니다.
#   1. 각 시나리오의 각 턴에서 agent_invoker 호출
#   2. CloudWatch 수집을 위해 evaluation_delay_seconds 동안 대기
#   3. 평가 서비스에 span 제출
#   4. 집계 결과 반환

eval_result = runner.run(
    config=config,
    dataset=dataset,
    agent_invoker=agent_invoker,
    span_collector=span_collector,
)

completed = sum(1 for sr in eval_result.scenario_results if sr.status == "COMPLETED")
failed = sum(1 for sr in eval_result.scenario_results if sr.status == "FAILED")
print(
    f"\nEvaluation complete: {completed} completed, {failed} failed out of {len(eval_result.scenario_results)} scenarios."
)

### 6c. 결과 확인

In [ ]:
def display_runner_results(eval_result) -> None:
    """OnDemandEvaluationDatasetRunner 결과를 시나리오별 Markdown 표로 표시합니다."""
    for sr in eval_result.scenario_results:
        if sr.status == "FAILED":
            display(Markdown(f"**Scenario `{sr.scenario_id}`** — FAILED: {sr.error}"))
            continue

        rows = ["| Evaluator | Value | Label | Explanation |", "|---|---|---|---|"]
        for er in sr.evaluator_results:
            for res in er.results:
                value = str(res.get("value", res.get("score", "N/A")))
                lbl = str(res.get("label", res.get("rating", "")))
                explanation = (res.get("explanation", "") or "")[:130].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:130]
                rows.append(f"| `{er.evaluator_id[:40]}` | {value} | {lbl} | {explanation} |")

        md = f"### Scenario: `{sr.scenario_id}`\n\n" + "\n".join(rows)
        display(Markdown(md))


display_runner_results(eval_result)

In [ ]:
# 집계 요약: 모든 시나리오의 evaluator별 평균 점수
from collections import defaultdict

scores_by_evaluator = defaultdict(list)
for sr in eval_result.scenario_results:
    if sr.status != "COMPLETED":
        continue
    for er in sr.evaluator_results:
        for res in er.results:
            if "value" in res and res["value"] is not None and not res.get("errorCode"):
                scores_by_evaluator[er.evaluator_id].append(float(res["value"]))

print("\nEvaluator Summary (average score across all scenarios)")
print("=" * 60)
for evaluator_id, scores in sorted(scores_by_evaluator.items()):
    avg = sum(scores) / len(scores)
    print(f"  {evaluator_id:<45} avg={avg:.2f}  (n={len(scores)})")

### 6d. 파일에 결과 저장

In [ ]:
import os
from datetime import datetime

os.makedirs("results", exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
results_path = f"results/groundtruth_eval_{timestamp}.json"

with open(results_path, "w") as f:
    json.dump(eval_result.model_dump(), f, indent=2, default=str)

print(f"Results saved to: {results_path}")

## 7단계: 배치 평가

`BatchEvaluationRunner`는 **단일 API 호출로 여러 세션을 평가**하고 evaluator별 집계 점수를 얻을 때 적합합니다. 클라이언트 측에서 평가를 실행하는 `OnDemandEvaluationDatasetRunner`와 달리, 배치 평가는 모든 세션 ID를 서비스에 제출하고 완료될 때까지 폴링합니다.

다음과 같은 경우에 적합합니다.
- 다수의 세션에 대한 기준선 측정
- prompt 또는 도구 변경 전후 비교
- 시나리오별 세부 정보보다 집계 지표가 필요한 프로덕션 모니터링

6단계와 동일한 `dataset` 및 `agent_invoker`를 재사용합니다. Runner는 각 시나리오에서 에이전트를 호출하고 CloudWatch 수집을 기다린 다음, 배치 평가 작업을 제출하고 완료될 때까지 폴링합니다.

In [ ]:
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig,
    BatchEvaluatorConfig,
    CloudWatchDataSourceConfig,
)

# 이전 셀의 에이전트 구성 재사용
SERVICE_NAME = f"{AGENT_ID}.DEFAULT"
CW_LOG_GROUP_BATCH = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"

print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP_BATCH}")

In [ ]:
import uuid

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, CW_LOG_GROUP_BATCH],
    ingestion_delay_seconds=180,
)

batch_config = BatchEvaluationRunConfig(
    batch_evaluation_name=f"gt_batch_{uuid.uuid4().hex[:8]}",
    evaluator_config=BatchEvaluatorConfig(
        evaluator_ids=[
            "Builtin.Correctness",
            "Builtin.GoalSuccessRate",
            "Builtin.TrajectoryExactOrderMatch",
        ]
    ),
    data_source=batch_data_source,
    polling_timeout_seconds=1800,
    polling_interval_seconds=30,
)

# 6단계와 동일한 dataset 및 agent_invoker 재사용
print(f"Batch evaluation name: {batch_config.batch_evaluation_name}")
print(f"Evaluators: {batch_config.evaluator_config.evaluator_ids}")
print(f"Dataset scenarios: {len(dataset.scenarios)}")
print("Starting batch evaluation (this may take several minutes) ...")

batch_runner = BatchEvaluationRunner(region=REGION)
batch_result = batch_runner.run_dataset_evaluation(
    config=batch_config,
    dataset=dataset,
    agent_invoker=agent_invoker,
)

print(f"\nBatch ID : {batch_result.batch_evaluation_id}")
print(f"Status   : {batch_result.status}")

In [ ]:
# --- 배치 평가 결과 ---
print(f"Batch ID:  {batch_result.batch_evaluation_id}")
print(f"ARN:       {batch_result.batch_evaluation_arn}")
print(f"Status:    {batch_result.status}")
print(f"Created:   {batch_result.created_at}")

if batch_result.evaluation_results:
    ev = batch_result.evaluation_results
    # 세션 수준 개수
    print(f"\nSessions completed: {ev.number_of_sessions_completed}")
    print(f"Sessions failed:    {ev.number_of_sessions_failed}")
    print(f"Total sessions:     {ev.total_number_of_sessions}")
    if ev.evaluator_summaries:
        print("\nPer-evaluator scores:")
        for es in ev.evaluator_summaries:
            eid = es.evaluator_id or "unknown"
            score = (
                f"{es.statistics.average_score:.3f}"
                if es.statistics and es.statistics.average_score is not None
                else "N/A"
            )
            evaluated = es.total_evaluated or 0
            failed = es.total_failed or 0
            print(f"  {eid:<40} score={score}  (evaluated={evaluated}, failed={failed})")
else:
    print("\nNo aggregated evaluation results returned.")

if batch_result.error_details:
    print(f"\nError details: {batch_result.error_details}")

# CloudWatch에서 세션별 세부 정보 가져오기(선택 사항)
if batch_result.output_data_config:
    events = batch_runner.fetch_evaluation_events(batch_result)
    print(f"\nPer-session evaluation events: {len(events)}")
    for ev in events[:5]:  # 처음 5개 표시
        attrs = ev.get("attributes", {})
        print(f"  session: {attrs.get('session.id', '')[:40]}")
        print(f"  evaluator: {attrs.get('gen_ai.evaluation.name')}")
        print(f"  score: {attrs.get('gen_ai.evaluation.score.value')}")
        print(f"  label: {attrs.get('gen_ai.evaluation.score.label')}")
        print()

## 시뮬레이션 멀티턴 평가

위 평가 기법은 스크립트로 작성된 사용자 입력이 포함된 사전 정의 시나리오를 사용합니다.
수작업을 줄여 데이터 세트를 구성하는 또 다른 방법은 사용자를 시뮬레이션해 시뮬레이션
데이터 세트를 만드는 것입니다. 사용자 시뮬레이션은 LLM 기반 actor가 에이전트와 상호 작용하는
최종 사용자 역할을 수행하게 합니다. actor의 프로필과 목표를 정의하면 목표를 달성하거나 턴 제한에
도달할 때까지 actor가 에이전트와 멀티턴 대화를 진행합니다. 자세히 알아보고 테스트하려면 다음
Notebook을 참조하세요.

**[`Strands-AgentCore-ShoppingConcierge.ipynb`](../03-advanced/02-simulating-agent-interactions/Strands-AgentCore-ShoppingConcierge.ipynb)**

해당 Notebook은 Shopping Concierge agent를 배포하고, 5가지 시뮬레이션 고객 시나리오
(헤드폰 구매, 주문 추적, 반품, 여러 품목 장바구니, 예산 적합성)를 실행한 뒤 단일
`StartBatchEvaluation` 호출로 모든 세션의 점수를 산출합니다. actor에는 `boto3 >= 1.43.0`과
Bedrock Converse API만 사용합니다.

## 8단계: 정리

지속적인 비용이 발생하지 않도록 완료 후 agent runtime 엔드포인트를 삭제하세요.

In [ ]:
# agent runtime을 삭제하려면 주석을 해제하세요
# cp = boto3.client("bedrock-agentcore-control", region_name=REGION)
# cp.delete_agent_runtime(agentRuntimeId=AGENT_ID)
# print("Agent runtime deleted.")

print("Cleanup skipped. Uncomment the cell above to delete the agent runtime.")

### 핵심 요점

| | EvaluationClient | OnDemandEvaluationDatasetRunner |
|---|---|---|
| **사용 시점** | 기존 세션이 있는 경우 | 테스트 데이터 세트가 있는 경우 |
| **적합한 용도** | 사후 분석, 디버깅 | 회귀 테스트, CI/CD |
| **입력** | session_id | 시나리오 데이터 세트 |

- **배치 평가**(`BatchEvaluationRunner`)는 단일 서비스 측 작업에서 여러 세션의 점수를 산출하며, 집계 지표 및 변경 전후 비교에 적합합니다.

### 기본 제공 evaluator 참조

| Evaluator | 수준 | 필요한 Ground Truth |
|---|---|---|
| `Builtin.Correctness` | TRACE | `expected_response` |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` |
| `Builtin.TrajectoryExactOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryInOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryAnyOrderMatch` | SESSION | `expected_trajectory` |


### 사용자 지정 evaluator의 Ground Truth placeholder

사용자 지정 LLM-as-a-Judge evaluator는 `instructions`의 placeholder를 통해 Ground Truth를 참조합니다.

| 수준 | Placeholder | 가져오는 위치 |
|---|---|---|
| TRACE | `{assistant_turn}` | 에이전트의 실제 응답 |
| TRACE | `{expected_response}` | `ReferenceInputs.expected_response` |
| TRACE | `{context}` | 세션 컨텍스트 |
| SESSION | `{actual_tool_trajectory}` | 에이전트가 호출한 도구 |
| SESSION | `{expected_tool_trajectory}` | `ReferenceInputs.expected_trajectory` |
| SESSION | `{assertions}` | `ReferenceInputs.assertions` |
| SESSION | `{available_tools}` | 에이전트가 사용할 수 있는 도구 |